# TP1 — Du modèle au service web : construire l'API
## Fil rouge : Churn Predictor (Session 4)

**Durée estimée : 1h45**

> ⚠️ **Format particulier** : ce notebook est un **manuel pas-à-pas**, pas un TP exécutable. Vous travaillez en parallèle dans **VS Code** sur le repo `S4_churn_api_squelette/`. Chaque section du notebook explique :
> - le fichier à modifier
> - ce que fait ce fichier dans l'architecture
> - le code attendu (à copier dans VS Code)
> - les pièges à éviter

### 🎯 Objectifs d'apprentissage
À la fin de ce TP vous saurez :
- Concevoir une **architecture de service ML** : separation of concerns (`schemas` / `model` / `routes`).
- Écrire des **schemas Pydantic** stricts (Literal, Field bounds) — la première ligne de défense en prod.
- Charger un modèle **une seule fois au démarrage** via le pattern `lifespan` de FastAPI.
- Construire des **endpoints REST** propres : `/predict`, `/predict-batch`, `/health`, `/model-info`.
- Lancer le serveur avec `uvicorn` et tester via **Swagger UI**.

### 📦 Pré-requis
- Le squelette `S4_churn_api_squelette/` est dézippé sur votre poste.
- VS Code (ou IDE équivalent) ouvert dessus.
- Un environnement Python 3.10+ avec `pip install -r requirements.txt` lancé.
- `artifacts/best_model.joblib` et `artifacts/manifest.json` présents (issus de S2 — fournis dans le squelette en backup).


## 1. L'architecture en 60 secondes

```
churn_api/
├── app/
│   ├── __init__.py
│   ├── config.py        ← chemins, métadonnées API (CONSTANTES)
│   ├── schemas.py       ← Pydantic models (validation I/O)
│   ├── model.py         ← chargement modèle + prédiction (LOGIQUE MÉTIER)
│   └── main.py          ← FastAPI app + endpoints (ROUTES)
├── tests/
│   └── test_api.py      ← pytest avec TestClient
├── artifacts/
│   ├── best_model.joblib
│   └── manifest.json
└── requirements.txt
```

### 🧠 Pourquoi cette séparation ?

| Fichier | Rôle | Pourquoi séparé |
|---------|------|-----------------|
| `schemas.py` | Définit *quoi* on accepte/retourne | Tests faciles, modification d'API sans toucher au modèle |
| `model.py` | Encapsule *comment* on prédit | Réutilisable hors API (script batch, notebook…) |
| `main.py` | Wire les routes HTTP | Chaque endpoint = 5-10 lignes max |
| `config.py` | Constantes globales | 1 endroit pour changer un chemin |

🧠 **Réflexe Tech Lead** : un junior met tout dans `main.py`. Un Tech Lead sépare. Le code de demain le remerciera (testable, refactorable, lisible).


## 2. `app/schemas.py` — la validation Pydantic

⏱ **Cible : 15 minutes**

### Pourquoi c'est ici qu'on commence
Pydantic est votre **première ligne de défense** en prod. Chaque requête passe par ses validateurs avant de toucher la moindre logique métier. Mieux votre schema est strict, moins vous aurez de bugs en aval.

### Ce qu'il faut écrire

Dans `app/schemas.py`, vous devez déclarer **6 classes** :

| Classe | Rôle |
|--------|------|
| `ClientFeatures` | Une fiche client en input (les 19 features Telco brutes) |
| `PredictionResponse` | La réponse pour 1 client |
| `BatchPredictionRequest` | Input pour le batch |
| `BatchPredictionResponse` | Output pour le batch |
| `HealthResponse` | Pour `/health` |
| `ModelInfoResponse` | Pour `/model-info` |

### Pattern à suivre pour les enums : `Literal`

Pour une enum stricte (Pydantic refuse toute autre valeur), on utilise `typing.Literal` :

```python
from typing import Literal
from pydantic import BaseModel, Field

class ClientFeatures(BaseModel):
    Contract: Literal["Month-to-month", "One year", "Two year"]
    SeniorCitizen: int = Field(ge=0, le=1)
    tenure: int = Field(ge=0, le=120)
    MonthlyCharges: float = Field(ge=0, le=500)
```

### ⚠️ Pièges à éviter
- ❌ `Contract: str` → accepte n'importe quoi, y compris des typos.
- ❌ `tenure: int` sans borne → quelqu'un peut envoyer `tenure=999999`.
- ❌ Oublier `gender` ou `Partner` → request 422 incompréhensible côté client.
- ❌ Mettre `MonthlyCharges` en `int` → "80.5 €" devient "80".

### 💡 Bonus : l'exemple Swagger

Ajoutez `model_config = ConfigDict(json_schema_extra={"example": {...}})` dans `ClientFeatures`. L'exemple sera rempli automatiquement dans Swagger UI → vos collègues peuvent cliquer "Try it out" sans rien chercher.

### Validation manuelle

Une fois `schemas.py` complété, dans un terminal Python :
```python
from app.schemas import ClientFeatures
ClientFeatures(gender="Female", SeniorCitizen=0, ...)  # avec tous les champs
# Doit ne pas lever d'erreur

ClientFeatures(gender="Other", ...)  # doit lever ValidationError
```

> ✅ **Critère de validation** : `from app.schemas import ClientFeatures` ne plante pas, et créer une instance avec un champ `Contract="Five years"` lève une `ValidationError`.


## 3. `app/model.py` — le service de prédiction

⏱ **Cible : 15 minutes**

### Le rôle de ce fichier
Encapsuler **tout** ce qui touche au modèle (chargement, feature engineering, prédiction) **hors de FastAPI**. Comme ça :
- On peut utiliser `ModelService` en script batch (pas que en HTTP).
- On peut le tester sans démarrer un serveur.
- Si on change FastAPI pour Flask demain, on ne touche pas à ce fichier.

### Ce qu'il faut écrire

Deux choses :
1. La fonction `add_engineered_features(df)` — **identique** à celle de S2 (les 4 features ingénierées).
2. La classe `ModelService` qui :
   - charge `best_model.joblib` et `manifest.json` au `__init__`
   - expose `predict_one(dict) -> dict` et `predict_batch(list[dict]) -> list[dict]`

### ⚠️ Le piège #1 absolu : sync entre training et inférence

> Si la fonction `add_engineered_features` du service **diffère** de celle utilisée à l'entraînement, le modèle reçoit des features dans un ordre/format différent et **prédit n'importe quoi**, sans erreur visible.

Solution propre : extraire `add_engineered_features` dans un module partagé (`features.py`) importé à l'entraînement ET en inférence. Pour cette session on duplique (manuel sync), mais en prod : code partagé.

### Squelette mental

```python
class ModelService:
    def __init__(self, model_path, manifest_path):
        self.model = joblib.load(model_path)
        self.manifest = json.loads(Path(manifest_path).read_text())
        self.threshold = float(self.manifest["threshold"])

    def predict_one(self, features_dict: dict) -> dict:
        df = pd.DataFrame([features_dict])
        df = add_engineered_features(df)
        proba = float(self.model.predict_proba(df)[0, 1])
        return {
            "churn_probability": proba,
            "churn_predicted": int(proba >= self.threshold),
            "threshold_used": self.threshold,
        }
```

### ⚠️ Pièges fréquents
- `predict_proba(df)[:, 1]` : on prend la **2ème colonne** (probabilité de la classe 1 = churn).
- `int(proba >= threshold)` : `bool` n'est pas JSON-serializable proprement, `int` l'est.
- `pd.DataFrame([features_dict])` : ne pas oublier les `[]` (on construit un DataFrame d'une ligne).

> ✅ **Critère de validation** : depuis un terminal Python à la racine du projet :
> ```python
> from app.model import ModelService
> from app.config import MODEL_PATH, MANIFEST_PATH
> svc = ModelService(MODEL_PATH, MANIFEST_PATH)
> result = svc.predict_one({"gender": "Female", "SeniorCitizen": 0, ...})  # avec tous les champs
> print(result)  # doit afficher un dict avec churn_probability ∈ [0, 1]
> ```


## 4. `app/main.py` — les endpoints FastAPI

⏱ **Cible : 25 minutes**

### Architecture des routes

| Méthode | Path | Réponse | Rôle |
|---------|------|---------|------|
| GET | `/` | 307 redirect | Redirige vers `/docs` |
| GET | `/health` | `HealthResponse` | Liveness probe (Kubernetes, ECS) |
| GET | `/model-info` | `ModelInfoResponse` | Métadonnées du modèle |
| POST | `/predict` | `PredictionResponse` | Prédiction sur 1 client |
| POST | `/predict-batch` | `BatchPredictionResponse` | Prédiction sur N clients |

### Le pattern critique : `lifespan`

🚨 **Ne JAMAIS charger le modèle dans le handler de la route**. Avec `joblib.load()` à chaque requête, vous transformez votre API à 1 ms en API à 1 seconde.

✅ **Bon pattern** : charger le modèle **une seule fois au démarrage** via le `lifespan` context manager :

```python
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup
    app.state.model_service = ModelService(MODEL_PATH, MANIFEST_PATH)
    yield
    # Shutdown (optionnel)

app = FastAPI(lifespan=lifespan)
```

Puis dans chaque route, accéder à `app.state.model_service`.

### Pattern de gestion d'erreur

```python
@app.post("/predict", response_model=PredictionResponse)
def predict(client: ClientFeatures):
    try:
        result = app.state.model_service.predict_one(client.model_dump())
        return PredictionResponse(**result)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {exc}")
```

🧠 **Pourquoi `try/except`** ? Pydantic gère déjà les erreurs de validation (→ 422). Mais si le modèle plante en interne (numpy nan, dimension mismatch…), on veut un 500 propre avec un message, pas un crash silencieux.

### ⚠️ Pièges fréquents

| Piège | Symptôme | Fix |
|-------|----------|-----|
| Charger le modèle au top-level (hors `lifespan`) | Ça marche mais le modèle est chargé même pour un import (test) | Utiliser `lifespan` |
| Oublier `response_model=...` | Swagger ne sait pas le format de retour | Toujours préciser |
| Retourner `client.dict()` (Pydantic v1) | DeprecationWarning | Utiliser `client.model_dump()` (Pydantic v2) |
| Path `/predict-batch` vs `/predict_batch` | Convention REST = kebab-case | `predict-batch` |
| Pas de tag (catégorie) sur les routes | Swagger UI plat | `tags=["predict"]` pour grouper |

### Validation manuelle

Une fois `main.py` complété :
```bash
uvicorn app.main:app --reload
```

Vous devez voir dans le terminal :
```
INFO:     Started server process [...]
INFO:     Waiting for application startup.
✅ Model loaded: HGBT_tuned
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000
```

Puis ouvrez **http://localhost:8000/docs** dans votre navigateur. Vous devez voir Swagger UI avec vos 5 endpoints.

> ✅ **Critère de validation** : Swagger affiche tous les endpoints, et "Try it out" sur `/predict` avec l'exemple Pydantic retourne 200 + un `churn_probability`.


## 5. Lancer & tester avec Swagger

⏱ **Cible : 15 minutes**

### 5.1 Démarrer le serveur
```bash
uvicorn app.main:app --reload
```
Le flag `--reload` redémarre le serveur à chaque modification de fichier — pratique en dev.

### 5.2 Tester via Swagger UI
1. Ouvrir http://localhost:8000/docs
2. Cliquer sur `POST /predict` → "Try it out"
3. L'exemple Pydantic est pré-rempli (si vous avez ajouté `model_config`)
4. Cliquer "Execute"
5. Lire la réponse : 200 + un JSON avec `churn_probability`

### 5.3 Tester via `curl`
```bash
curl -X POST http://localhost:8000/health
```

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "gender": "Female", "SeniorCitizen": 0,
    "Partner": "Yes", "Dependents": "No",
    "tenure": 12, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No",
    "StreamingTV": "No", "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 80.0, "TotalCharges": 960.0
  }'
```

### 5.4 Tests d'erreur volontaires (apprentissage critique)

Que se passe-t-il si vous envoyez :
- Un payload sans `tenure` ?
- `Contract: "Five years"` ?
- `tenure: -3` ?
- `MonthlyCharges: "abc"` ?

→ FastAPI doit retourner un **422 Unprocessable Entity** avec un message clair indiquant le champ fautif. Si vous obtenez un 500 ou un crash, votre schema Pydantic est trop laxiste.

> ✅ **Critère de validation final TP1** : 5 endpoints fonctionnels dans Swagger, payload valide → 200, payload invalide → 422 avec message.

---

## 🛑 Pause de 15 minutes

Au retour : on automatise les tests avec pytest et on regarde sous le capot d'OpenAPI.
